In [ ]:
# Imports
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.visualization.wcsaxes import WCSAxes
from astropy.coordinates import SkyCoord, FK5
from spectral_cube import SpectralCube
from velocity_tools import extract_streamline, gradient_descent, stream_lines_grad
from velocity_tools import stream_lines # won't use this directly, but needed to compare with stream_lines_grad
import os
import numpy as np
import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax import value_and_grad
import pandas as pd
import optax

import warnings
warnings.filterwarnings('ignore', message='.*PV2_1.*')
warnings.filterwarnings('ignore', message='.*PV2_2.*')
warnings.filterwarnings('ignore', message='.*TIMESYS.*')

cubefile = 'test_data/IRAS2A/D2CO_streamer_cluster_data.fits'
n_points_values = [5, 10, 15, 20]

In [ ]:
def plot_data_and_extracted_streamline(pc_coords, pc_means, pc_stds):
    # plot the observed data points as a scatter
    plt.scatter(pc_coords[0], pc_coords[1], s=1, alpha=0.3, color='grey', label='Point cloud')
    # plot the extracted 1D streamline with error bars
    plt.errorbar(pc_means[0], pc_means[1], xerr=pc_stds[0], yerr=pc_stds[1], fmt='o-', label='Extracted 1D Streamline', color='red')
    # plot the star
    plt.scatter(0, 0, marker='*', s=100, color='yellow', edgecolor='black', label='Star', zorder=10)
    plt.xlabel('RA Offset (arcsec)')
    plt.ylabel('Dec Offset (arcsec)')
    # flip the x axis to match the astronomical convention (RA increases to the left)
    plt.gca().invert_xaxis()
    plt.legend()
    plt.title('Extracted 1D Streamer emission')


def extract_point_cloud(streamer_cube):
    print('Starting reduction')
    nz, ny, nx = streamer_cube.shape

    # create coordinate arrays for RA and Dec in arcsec
    y_indices, x_indices = np.mgrid[0:ny, 0:nx]
    world_coords = streamer_cube.wcs.celestial.pixel_to_world_values(x_indices.ravel(), y_indices.ravel())
    ra_coords = (world_coords[0].reshape(ny, nx) - streamer_cube.header['CRVAL1']) * 60 * 60
    ra_coords = ra_coords * np.cos(streamer_cube.header['CRVAL2'] * np.pi / 180) # cos(dec) correct for declination. in arcsec
    dec_coords = (world_coords[1].reshape(ny, nx) - streamer_cube.header['CRVAL2']) * 60 * 60 # in arcsec

    # create velocity array in km/s
    #TODO: fix this to use spectral_axis and WCS instead of header keywords, to be more robust
    v_coords = streamer_cube.spectral_axis.to(u.km/u.s).value - (streamer_cube.header['CRVAL3']*1e-3)

    print('Created coordinate arrays')

    # get data and mask
    pcloud = np.array(streamer_cube)
    rms_mask = ~np.isnan(pcloud)
    flux = pcloud[rms_mask]

    # get indices of valid points in pc
    pc_indices = np.indices(pcloud.shape) # indices of all points in pc
    pc_z = pc_indices[0][rms_mask] # z indices of points in pc
    pc_y = pc_indices[1][rms_mask] # y indices of points in pc
    pc_x = pc_indices[2][rms_mask] # x indices of points in pc

    print('Got point cloud with', len(flux), 'points')

    # extract coordinates of valid points using the arrays above
    pc_ra = ra_coords[pc_y, pc_x]
    pc_dec = dec_coords[pc_y, pc_x]
    pc_v = v_coords[pc_z]
    pc_coords = np.array([pc_ra, pc_dec, pc_v]) # shape (3, n_points) 

    return pc_coords, flux


Prepare cube

In [ ]:
# get the spectralcube object from the data using spectral-cube
hdu = fits.open(cubefile)[0]
cube = SpectralCube.read(hdu).with_spectral_unit(u.km/u.s, rest_value=hdu.header['RESTFRQ']*u.Hz)

# TODO: this streamer extraction should be replaced with clustering-based streamer extraction

# extract the subcube with the streamer (we got this from the tipsy tutorial, no need to plot)
## Limits for extracting subcube with streamer
'''
vmin = 7    # Min. vel. of streamer
vmax = 10
xmin = -3   # Min R.A. offset (arcsec) to consider for streamer
xmax = -1
ymin = -3   # Min. Decl. offset (arcsec) to consider for streamer
ymax = 0.5
rms_thresh = 4  # sigma threshold for streamer
'''

vmin = 6
vmax = 8
xmin = -5
xmax = 5
ymin = -12
ymax = 0.5
rms_thresh = 4

# extract the streamer subcube
info_header = cube.header  # header of the cube, contains required information
x_conv_fac = 1/60/60/info_header['CDELT1']
xmin_p = int((xmin*x_conv_fac)+info_header['CRPIX1'])
xmax_p = int((xmax*x_conv_fac)+info_header['CRPIX1'])
y_conv_fac = 1/60/60/info_header['CDELT2']
ymin_p = int((ymin*y_conv_fac)+info_header['CRPIX2'])
ymax_p = int((ymax*y_conv_fac)+info_header['CRPIX2'])
vunit = cube.spectral_axis.unit
## Note: a check can be added to see if requested limits are within the limits of the cube itself

streamer_cubev = cube.spectral_slab(vmin*vunit,vmax*vunit)    # Selecting velocities
streamer_cubevc = streamer_cubev[:,min(ymin_p,ymax_p):max(ymin_p,ymax_p)   
                    ,min(xmin_p,xmax_p):max(xmin_p,xmax_p)]   # Selecting pixels
#     print(min(ymin_p,ymax_p),max(ymin_p,ymax_p),min(xmin_p,xmax_p),max(xmin_p,xmax_p)) 
streamer_cube = streamer_cubevc.with_mask(streamer_cubevc > rms_thresh*streamer_cubevc.mad_std())  # Removing low flux values 


### 1. Original Method

In [ ]:
def compute_method1(streamer_cube, n_points):
    # Original reduction from cube directly
    _, pc_means1, pc_stds1 = extract_streamline.reduce_to_1D(
        streamer_cube,
        n_elements=n_points,
    )
    return pc_means1, pc_stds1

### 2. New method - bin in 3D (RA, Dec, velocity)

In [ ]:
def compute_method2(pc_coords, flux, n_points):
    # Bin point cloud in 3D spherical bins
    distance_metric = np.sqrt(pc_coords[0]**2 + pc_coords[1]**2 + pc_coords[2]**2)
    partitions = np.percentile(distance_metric, np.linspace(0, 100, n_points + 1))

    pc_means2 = np.zeros((3, n_points))
    pc_stds2 = np.zeros((3, n_points))
    for i in range(n_points):
        distance_indices = (distance_metric > partitions[i]) & (distance_metric <= partitions[i + 1])
        if not np.any(distance_indices):
            center = 0.5 * (partitions[i] + partitions[i + 1])
            idx = np.argmin(np.abs(distance_metric - center))
            pc_means2[:, i] = pc_coords[:, idx]
            pc_stds2[:, i] = 0.0
            continue

        pc_means2[:, i] = np.average(
            pc_coords.T[distance_indices],
            axis=0,
            weights=flux[distance_indices],
        )
        pc_stds2[:, i] = np.sqrt(
            np.average(
                (pc_coords.T[distance_indices] - pc_means2[:, i])**2,
                axis=0,
                weights=flux[distance_indices],
            )
        )

    # Flip arrays so that they go from large to small distance (towards star)
    pc_means2 = pc_means2[:, ::-1]
    pc_stds2 = pc_stds2[:, ::-1]
    return pc_means2, pc_stds2

### 3. New method - equal projected-radius bins

In [ ]:
def compute_method3(pc_coords, flux, n_points):
    # Use projected radius bins with equal radial width from min(r_proj) to max(r_proj)
    proj_radius = np.sqrt(pc_coords[0]**2 + pc_coords[1]**2)
    r_edges = np.linspace(proj_radius.min(), proj_radius.max(), n_points + 1)

    pc_means3 = np.zeros((3, n_points))
    pc_stds3 = np.zeros((3, n_points))

    for i in range(n_points):
        if i == 0:
            in_bin = (proj_radius >= r_edges[i]) & (proj_radius <= r_edges[i + 1])
        else:
            in_bin = (proj_radius > r_edges[i]) & (proj_radius <= r_edges[i + 1])

        if not np.any(in_bin):
            r_center = 0.5 * (r_edges[i] + r_edges[i + 1])
            idx = np.argmin(np.abs(proj_radius - r_center))
            pc_means3[:, i] = pc_coords[:, idx]
            pc_stds3[:, i] = 0.0
            continue

        bin_coords = pc_coords.T[in_bin]
        bin_weights = flux[in_bin]
        mean_vec = np.average(bin_coords, axis=0, weights=bin_weights)
        var_vec = np.average((bin_coords - mean_vec) ** 2, axis=0, weights=bin_weights)

        pc_means3[:, i] = mean_vec
        pc_stds3[:, i] = np.sqrt(var_vec)

    return pc_means3, pc_stds3

### 4. Possible more complicated idea: get 'density ridge' similar to in Chen+ 2020

### Use methods, plot

In [ ]:
# Extract point cloud once and reuse it for methods 2 and 3
pc_coords_base, flux_base = extract_point_cloud(streamer_cube)

for n_points in n_points_values:
    pc_means1, pc_stds1 = compute_method1(streamer_cube, n_points)
    pc_means2, pc_stds2 = compute_method2(pc_coords_base, flux_base, n_points)
    pc_means3, pc_stds3 = compute_method3(pc_coords_base, flux_base, n_points)

    plt.figure(figsize=(6, 6))

    # point cloud background
    plt.scatter(pc_coords_base[0], pc_coords_base[1], s=4, alpha=0.2, color='lightgrey', label='Point cloud')

    # method 1: original method
    plt.errorbar(
        pc_means1[0],
        pc_means1[1],
        xerr=pc_stds1[0],
        yerr=pc_stds1[1],
        fmt='o-',
        capsize=2,
        color='tab:red',
        label='Method 1: Original',
    )

    # method 2: 3D binning method
    plt.errorbar(
        pc_means2[0],
        pc_means2[1],
        xerr=pc_stds2[0],
        yerr=pc_stds2[1],
        fmt='s-',
        capsize=2,
        color='tab:blue',
        label='Method 2: 3D bins',
    )

    # method 3: equal projected-radius bins
    plt.errorbar(
        pc_means3[0],
        pc_means3[1],
        xerr=pc_stds3[0],
        yerr=pc_stds3[1],
        fmt='^-',
        capsize=2,
        color='tab:green',
        label='Method 3: Equal proj-radius bins',
    )

    # plot the star
    plt.scatter(0, 0, marker='*', s=120, color='yellow', edgecolor='black', label='Star', zorder=10)

    plt.xlabel('RA Offset (arcsec)')
    plt.ylabel('Dec Offset (arcsec)')
    plt.gca().invert_xaxis()
    plt.title(f'Comparison of Extracted 1D Streamlines (n_points={n_points})')
    plt.legend()
    plt.tight_layout()
    plt.show()